#### Setup

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, LongType, FloatType, ArrayType
)
import requests
import json
import itertools
import pandas as pd
import random
from datetime import datetime, timedelta

In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Practice")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "1g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready. Shuffle partitions: 4")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 15:43:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3 ready. Shuffle partitions: 4


#### Exchange Rate data from Frankfurther API

In [3]:
fx_resp = requests.get(
    "https://api.frankfurter.dev/v1/2024-10-01..2025-03-01",
    params={"symbols": "USD,GBP,JPY,CHF,CAD,AUD,SEK,NOK,PLN,CZK"},
    timeout=15
)
fx_resp.raise_for_status()

In [4]:
fx_data = fx_resp.json()

In [5]:
print(list(fx_data.keys()))

first_date = list(fx_data["rates"].keys())[0]
print(f"\n{first_date}: {fx_data['rates'][first_date]}")

# Or just how many dates came back
print(f"\nTotal dates: {len(fx_data['rates'])}")

['amount', 'base', 'start_date', 'end_date', 'rates']

2024-10-01: {'AUD': 1.604, 'CAD': 1.4986, 'CHF': 0.9394, 'CZK': 25.272, 'GBP': 0.83193, 'JPY': 159.37, 'NOK': 11.7305, 'PLN': 4.2853, 'SEK': 11.3145, 'USD': 1.1086}

Total dates: 106


In [6]:
for date, rates in itertools.islice(fx_data["rates"].items(), 5):
    print(date, rates)

2024-10-01 {'AUD': 1.604, 'CAD': 1.4986, 'CHF': 0.9394, 'CZK': 25.272, 'GBP': 0.83193, 'JPY': 159.37, 'NOK': 11.7305, 'PLN': 4.2853, 'SEK': 11.3145, 'USD': 1.1086}
2024-10-02 {'AUD': 1.6048, 'CAD': 1.4932, 'CHF': 0.9388, 'CZK': 25.316, 'GBP': 0.83288, 'JPY': 160.26, 'NOK': 11.6735, 'PLN': 4.295, 'SEK': 11.3495, 'USD': 1.1071}
2024-10-03 {'AUD': 1.6124, 'CAD': 1.4945, 'CHF': 0.9387, 'CZK': 25.357, 'GBP': 0.84258, 'JPY': 161.98, 'NOK': 11.7195, 'PLN': 4.3058, 'SEK': 11.361, 'USD': 1.1039}
2024-10-04 {'AUD': 1.6121, 'CAD': 1.4952, 'CHF': 0.9394, 'CZK': 25.347, 'GBP': 0.83735, 'JPY': 161.69, 'NOK': 11.6845, 'PLN': 4.3145, 'SEK': 11.3375, 'USD': 1.1029}
2024-10-07 {'AUD': 1.6155, 'CAD': 1.4914, 'CHF': 0.9388, 'CZK': 25.346, 'GBP': 0.83918, 'JPY': 162.63, 'NOK': 11.656, 'PLN': 4.3215, 'SEK': 11.366, 'USD': 1.0982}


In [7]:
fx_rows = []
for date_str, rates in fx_data.get("rates",{}).items():
    for currency, rate in rates.items():
        fx_rows.append((date_str, fx_data["base"], currency, float(rate)))

exch_rt_df = spark.createDataFrame(
    fx_rows,
    schema=["rate_date", "base_currency", "target_currency", "rate"]
).withColumn("rate_date", F.to_date("rate_date"))

In [8]:
exch_rt_df.show(5)

+----------+-------------+---------------+-------+
| rate_date|base_currency|target_currency|   rate|
+----------+-------------+---------------+-------+
|2024-10-01|          EUR|            AUD|  1.604|
|2024-10-01|          EUR|            CAD| 1.4986|
|2024-10-01|          EUR|            CHF| 0.9394|
|2024-10-01|          EUR|            CZK| 25.272|
|2024-10-01|          EUR|            GBP|0.83193|
+----------+-------------+---------------+-------+
only showing top 5 rows



#### Country data from REST Countries API

In [9]:
countries_resp = requests.get(
    "https://restcountries.com/v3.1/all",
    params={
        "fields": "name,cca2,capital,region,subregion,population,area,currencies,languages,latlng"
    },
    timeout=20,
)
countries_resp.raise_for_status()

In [10]:
countries_raw = countries_resp.json()

In [11]:
countries_raw[0]

{'name': {'common': 'Zimbabwe',
  'official': 'Republic of Zimbabwe',
  'nativeName': {'bwg': {'official': 'Republic of Zimbabwe',
    'common': 'Zimbabwe'},
   'eng': {'official': 'Republic of Zimbabwe', 'common': 'Zimbabwe'},
   'kck': {'official': 'Republic of Zimbabwe', 'common': 'Zimbabwe'},
   'khi': {'official': 'Republic of Zimbabwe', 'common': 'Zimbabwe'},
   'ndc': {'official': 'Republic of Zimbabwe', 'common': 'Zimbabwe'},
   'nde': {'official': 'Republic of Zimbabwe', 'common': 'Zimbabwe'},
   'nya': {'official': 'Republic of Zimbabwe', 'common': 'Zimbabwe'},
   'sna': {'official': 'Republic of Zimbabwe', 'common': 'Zimbabwe'},
   'sot': {'official': 'Republic of Zimbabwe', 'common': 'Zimbabwe'},
   'toi': {'official': 'Republic of Zimbabwe', 'common': 'Zimbabwe'},
   'tsn': {'official': 'Republic of Zimbabwe', 'common': 'Zimbabwe'},
   'tso': {'official': 'Republic of Zimbabwe', 'common': 'Zimbabwe'},
   'ven': {'official': 'Republic of Zimbabwe', 'common': 'Zimbabwe'},
  

In [12]:
countries_raw_df = spark.createDataFrame(pd.DataFrame(countries_raw))
countries_raw_df.show(5)

+--------------------+--------------------+--------------------+-------------------+----+--------------+-------+---------------+--------+----------+
|                name|          currencies|           languages|             latlng|cca2|       capital| region|      subregion|    area|population|
+--------------------+--------------------+--------------------+-------------------+----+--------------+-------+---------------+--------+----------+
|{nativeName -> {s...|{ZWL -> {name -> ...|{sna -> Shona, nd...|      [-20.0, 30.0]|  ZW|      [Harare]| Africa|Southern Africa|390757.0|  17073087|
|{nativeName -> {g...|{AUD -> {name -> ...|{gil -> Gilbertes...|[1.41666666, 173.0]|  KI|[South Tarawa]|Oceania|     Micronesia|   811.0|    120740|
|{nativeName -> {e...|{GHS -> {name -> ...|    {eng -> English}|        [8.0, -2.0]|  GH|       [Accra]| Africa| Western Africa|238533.0|  33742380|
|{nativeName -> {k...|{KPW -> {name -> ...|     {kor -> Korean}|      [40.0, 127.0]|  KP|   [Pyongyang]|  

In [13]:
countries_raw_df.printSchema()

root
 |-- name: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- currencies: map (nullable = true)
 |    |-- key: string
 |    |-- value: map (valueContainsNull = true)
 |    |    |-- key: string
 |    |    |-- value: string (valueContainsNull = true)
 |-- languages: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- latlng: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- cca2: string (nullable = true)
 |-- capital: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- region: string (nullable = true)
 |-- subregion: string (nullable = true)
 |-- area: double (nullable = true)
 |-- population: long (nullable = true)



In [14]:
countries_df = (
    countries_raw_df
    .withColumn("country_name", F.col("name")["common"])
    .withColumn("official_name", F.col("name")["official"])
    .withColumn("country_code", F.col("cca2"))
    .withColumn("currency_codes", F.map_keys(F.col("currencies")))
    .withColumn("language_codes", F.map_keys(F.col("languages")))
).select("country_name", "official_name", "country_code", "currency_codes", "language_codes", "capital", "region", "subregion", "area", "population") 

In [15]:
countries_df.show(5)

+------------+--------------------+------------+--------------+--------------------+--------------+-------+---------------+--------+----------+
|country_name|       official_name|country_code|currency_codes|      language_codes|       capital| region|      subregion|    area|population|
+------------+--------------------+------------+--------------+--------------------+--------------+-------+---------------+--------+----------+
|    Zimbabwe|Republic of Zimbabwe|          ZW|         [ZWL]|[sna, ndc, khi, t...|      [Harare]| Africa|Southern Africa|390757.0|  17073087|
|    Kiribati|Independent and S...|          KI|    [AUD, KID]|          [gil, eng]|[South Tarawa]|Oceania|     Micronesia|   811.0|    120740|
|       Ghana|   Republic of Ghana|          GH|         [GHS]|               [eng]|       [Accra]| Africa| Western Africa|238533.0|  33742380|
| North Korea|Democratic People...|          KP|         [KPW]|               [kor]|   [Pyongyang]|   Asia|   Eastern Asia|120538.0|  25

In [16]:
d_languages_raw = countries_raw_df.select("languages")
d_languages = (d_languages_raw.select(F.explode("languages").alias("lang_code", "language")).distinct())
print(d_languages.count())
d_languages.show(5, truncate=False)

155
+---------+----------+
|lang_code|language  |
+---------+----------+
|khi      |Khoisan   |
|tsn      |Tswana    |
|tso      |Tsonga    |
|eng      |English   |
|gil      |Gilbertese|
+---------+----------+
only showing top 5 rows



In [17]:
d_currencies_raw = countries_raw_df.select("currencies")
d_currencies = (
    d_currencies_raw
    .select(F.explode("currencies").alias("currency_code", "currency_info"))
    .select(
        "currency_code",
        F.col("currency_info")["name"].alias("currency_name"),
        F.col("currency_info")["symbol"].alias("currency_symbol"),
    )
    .distinct()
)
print(d_currencies.count())
d_currencies.show(10, truncate=False)


170
+-------------+------------------+---------------+
|currency_code|currency_name     |currency_symbol|
+-------------+------------------+---------------+
|AUD          |Australian dollar |$              |
|BIF          |Burundian franc   |Fr             |
|AFN          |Afghan afghani    |؋              |
|BOB          |Bolivian boliviano|Bs.            |
|MMK          |Burmese kyat      |Ks             |
|TND          |Tunisian dinar    |د.ت            |
|UGX          |Ugandan shilling  |Sh             |
|KHR          |Cambodian riel    |៛              |
|MAD          |Moroccan dirham   |DH             |
|ALL          |Albanian lek      |L              |
+-------------+------------------+---------------+
only showing top 10 rows



#### Exchange Rate Data

In [18]:
live_resp = requests.get("https://open.er-api.com/v6/latest/EUR", timeout=15)
live_resp.raise_for_status()
live_data = live_resp.json()

In [19]:
live_rows = [
    ("EUR", currency, float(rate), live_data.get("time_last_update_utc", ""))
    for currency, rate in live_data.get("rates", {}).items()
]

live_rates = spark.createDataFrame(
    live_rows,
    schema=["base", "target", "rate", "last_updated"]
)

live_rates.cache()
print(f"live_rates: {live_rates.count()} rows")
live_rates.show(5)

live_rates: 166 rows
+----+------+----------+--------------------+
|base|target|      rate|        last_updated|
+----+------+----------+--------------------+
| EUR|   EUR|       1.0|Tue, 10 Mar 2026 ...|
| EUR|   AED|   4.25392|Tue, 10 Mar 2026 ...|
| EUR|   AFN| 73.161199|Tue, 10 Mar 2026 ...|
| EUR|   ALL| 95.777857|Tue, 10 Mar 2026 ...|
| EUR|   AMD|436.334694|Tue, 10 Mar 2026 ...|
+----+------+----------+--------------------+
only showing top 5 rows



#### JSON Placeholder Data

In [20]:
_users = requests.get("https://jsonplaceholder.typicode.com/users", timeout=10).json()
_posts = requests.get("https://jsonplaceholder.typicode.com/posts", timeout=10).json()
_comments = requests.get("https://jsonplaceholder.typicode.com/comments", timeout=10).json()

In [21]:
print(type(_comments))
print(len(_comments))

<class 'list'>
500


In [22]:
users = spark.createDataFrame([
    (u["id"], u["name"], u["username"], u["email"],
     u["address"]["city"], float(u["address"]["geo"]["lat"]),
     float(u["address"]["geo"]["lng"]), u["company"]["name"])
    for u in _users
], schema=["user_id", "name", "username", "email", "city", "lat", "lng", "company_name"])

posts = spark.createDataFrame([
    (p["id"], p["userId"], p["title"], p["body"].replace("\n", " "))
    for p in _posts
], schema=["post_id", "user_id", "title", "body"])

comments = spark.createDataFrame([
    (c["id"], c["postId"], c["name"], c["email"], c["body"].replace("\n", " "))
    for c in _comments
], schema=["comment_id", "post_id", "commenter_name", "commenter_email", "comment_body"])

In [23]:
print(f"users: {users.count()}, posts: {posts.count()}, comments: {comments.count()}")

users: 10, posts: 100, comments: 500


#### Country/Currency Transactions

In [24]:
import random
from datetime import datetime, timedelta

random.seed(42)

_country_currency = (
    countries_df
    .filter(F.col("population") > 1_000_000)
    .select("country_code", "region", F.explode("currency_codes").alias("currency_code"))
    .collect()
)

print(f"Country-currency combos available: {len(_country_currency)}")

_merchants = [
    ("Amazon", "retail"), ("Uber", "transport"), ("Netflix", "entertainment"),
    ("Carrefour", "grocery"), ("Shell", "fuel"), ("Glovo", "delivery"),
    ("Spotify", "subscription"), ("McDonalds", "dining"), ("Zara", "retail"),
    ("Bolt", "transport"), ("Lidl", "grocery"), ("IKEA", "retail"),
]
_statuses = ["completed"] * 5 + ["pending", "failed", "refunded"]

base_dt = datetime(2024, 10, 1)

txn_rows = []
for i in range(1, 1501):
    cc = random.choice(_country_currency)
    m_name, m_cat = random.choice(_merchants)
    fs = round(random.uniform(0.4, 0.99), 4) if random.random() < 0.06 else None

    txn_rows.append((
        f"TXN-{i:06d}",
        f"C-{random.randint(1, 150):04d}",
        m_name,
        m_cat,
        round(random.uniform(2.0, 700.0), 2),
        cc["currency_code"],
        cc["country_code"],
        cc["region"],
        random.choice(_statuses),
        (base_dt + timedelta(
            days=random.randint(0, 150),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59),
        )).strftime("%Y-%m-%d %H:%M:%S"),
        fs,
    ))

transactions = spark.createDataFrame(txn_rows, schema=StructType([
    StructField("txn_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("merchant_name", StringType()),
    StructField("category", StringType()),
    StructField("amount", DoubleType()),
    StructField("currency", StringType()),
    StructField("country_code", StringType()),
    StructField("region", StringType()),
    StructField("status", StringType()),
    StructField("txn_timestamp", StringType()),
    StructField("fraud_score", DoubleType()),
]))


print(f"transactions: {transactions.count()} rows")
transactions.show(5, truncate=False)

Country-currency combos available: 169
transactions: 1500 rows
+----------+-----------+-------------+---------+------+--------+------------+--------+---------+-------------------+-----------+
|txn_id    |customer_id|merchant_name|category |amount|currency|country_code|region  |status   |txn_timestamp      |fraud_score|
+----------+-----------+-------------+---------+------+--------+------------+--------+---------+-------------------+-----------+
|TXN-000001|C-0058     |Uber         |transport|99.4  |NPR     |NP          |Asia    |completed|2025-02-17 02:37:00|0.5623     |
|TXN-000002|C-0130     |Amazon       |retail   |422.21|KES     |KE          |Africa  |completed|2025-02-17 13:14:00|0.529      |
|TXN-000003|C-0002     |Bolt         |transport|531.65|ZMW     |ZM          |Africa  |completed|2025-01-17 10:17:00|NULL       |
|TXN-000004|C-0087     |Carrefour    |grocery  |73.34 |EGP     |EG          |Africa  |failed   |2024-10-25 11:54:00|NULL       |
|TXN-000005|C-0012     |Bolt      

### Data Inventory

In [25]:
print("=" * 60)
print("ALL DATA LOADED — Summary")
print("=" * 60)
print(f"  exch_rt_df     : {exch_rt_df.count():>5} rows  (ECB daily, 90 days, 10 currencies)")
print(f"  countries_df   : {countries_df.count():>5} rows  (all countries, live API)")
print(f"  d_currencies   : {d_currencies.count():>5} rows  (currency code/name/symbol)")
print(f"  d_languages    : {d_languages.count():>5} rows  (language code/name)")
print(f"  live_rates     : {live_rates.count():>5} rows  (EUR snapshot, 160+ currencies)")
print(f"  users          : {users.count():>5} rows  (JSONPlaceholder)")
print(f"  posts          : {posts.count():>5} rows  (JSONPlaceholder)")
print(f"  comments       : {comments.count():>5} rows  (JSONPlaceholder)")
print(f"  transactions   : {transactions.count():>5} rows  (refs real countries/currencies)")
print("=" * 60)

ALL DATA LOADED — Summary
  exch_rt_df     :  1060 rows  (ECB daily, 90 days, 10 currencies)
  countries_df   :   250 rows  (all countries, live API)
  d_currencies   :   170 rows  (currency code/name/symbol)
  d_languages    :   155 rows  (language code/name)
  live_rates     :   166 rows  (EUR snapshot, 160+ currencies)
  users          :    10 rows  (JSONPlaceholder)
  posts          :   100 rows  (JSONPlaceholder)
  comments       :   500 rows  (JSONPlaceholder)
  transactions   :  1500 rows  (refs real countries/currencies)


### Exercises

#### Exercise 1:
Using `transactions`:

1. Cast `txn_timestamp` (string) to `TimestampType`. Extract:
   - `txn_date` (DateType)
   - `txn_hour` (int, 0–23)

2. Filter: `status == "completed"` AND `amount > 25`

3. Add `amount_bucket`:

   | Condition | Bucket |
   |---|---|
   | amount < 20 | micro |
   | 20 ≤ amount < 100 | small |
   | 100 ≤ amount < 300 | medium |
   | amount ≥ 300 | large |

4. Select: `txn_id`, `customer_id`, `merchant_name`, `amount`, `currency`, `country_code`, `txn_date`, `txn_hour`, `amount_bucket`

Show 10 rows.


In [26]:
transactions.show(5, truncate=False)

+----------+-----------+-------------+---------+------+--------+------------+--------+---------+-------------------+-----------+
|txn_id    |customer_id|merchant_name|category |amount|currency|country_code|region  |status   |txn_timestamp      |fraud_score|
+----------+-----------+-------------+---------+------+--------+------------+--------+---------+-------------------+-----------+
|TXN-000001|C-0058     |Uber         |transport|99.4  |NPR     |NP          |Asia    |completed|2025-02-17 02:37:00|0.5623     |
|TXN-000002|C-0130     |Amazon       |retail   |422.21|KES     |KE          |Africa  |completed|2025-02-17 13:14:00|0.529      |
|TXN-000003|C-0002     |Bolt         |transport|531.65|ZMW     |ZM          |Africa  |completed|2025-01-17 10:17:00|NULL       |
|TXN-000004|C-0087     |Carrefour    |grocery  |73.34 |EGP     |EG          |Africa  |failed   |2024-10-25 11:54:00|NULL       |
|TXN-000005|C-0012     |Bolt         |transport|511.35|VES     |VE          |Americas|completed|2

In [27]:
transactions = (
    transactions
    .withColumn("txn_timestamp",F.to_timestamp(F.col("txn_timestamp")))
    ).withColumn("txn_date", F.to_date(F.col("txn_timestamp"))).withColumn("txn_hour", F.hour(F.col("txn_timestamp"))).filter((F.col("status")=="completed") & (F.col("amount") > 25))



In [28]:
transactions_bucketed = (
    transactions
    .withColumn(
        "amount_bucket",
        F.when(F.col("amount") < 20, "micro")
        .when(F.col("amount").between(20, 100), "small")
        .when(F.col("amount").between(101, 299), "medium")
        .when(F.col("amount") >= 300, "large")
        .otherwise("INVALID AMOUNT")
    )
    .select("txn_id", "customer_id", "merchant_name", "amount", "currency", "country_code", "txn_date", "txn_hour", "amount_bucket")
)

transactions_bucketed.show(10, truncate=False)
transactions_bucketed.explain()

+----------+-----------+-------------+------+--------+------------+----------+--------+-------------+
|txn_id    |customer_id|merchant_name|amount|currency|country_code|txn_date  |txn_hour|amount_bucket|
+----------+-----------+-------------+------+--------+------------+----------+--------+-------------+
|TXN-000001|C-0058     |Uber         |99.4  |NPR     |NP          |2025-02-17|2       |small        |
|TXN-000002|C-0130     |Amazon       |422.21|KES     |KE          |2025-02-17|13      |large        |
|TXN-000003|C-0002     |Bolt         |531.65|ZMW     |ZM          |2025-01-17|10      |large        |
|TXN-000005|C-0012     |Bolt         |511.35|VES     |VE          |2025-01-05|2       |large        |
|TXN-000006|C-0093     |Lidl         |404.99|EUR     |IE          |2024-10-12|21      |large        |
|TXN-000008|C-0019     |Carrefour    |427.17|XOF     |ML          |2025-02-14|23      |large        |
|TXN-000011|C-0068     |Lidl         |99.46 |SLE     |SL          |2025-02-27|13  

#### Exercise 2: Multi-Source Joins (12 min)

Build a unified transaction view.

**Step 1 — Country enrichment:**
Join `transactions` with `countries_df` on `country_code` to get `country_name`, `region`, `subregion`, `population`.

**Step 2 — Currency normalization:**
Join with `exch_rt_df` to convert amounts to EUR.

- `exch_rt_df` columns: `rate_date`, `base_currency` (EUR), `target_currency`, `rate`
- The rate is EUR → target. So: `amount_eur = amount / rate`
- Join on: `currency = target_currency` AND `txn_date = rate_date`
- ECB only covers ~10 currencies. Use a LEFT join so you don't lose rows.
- Transactions in uncovered currencies get `amount_eur = null`.

**Final columns:**
`txn_id`, `customer_id`, `merchant_name`, `category`, `amount`, `currency`, `amount_eur`, `country_name`, `region`, `population`, `status`, `txn_date`, `fraud_score`

**After you code it:**

- Count how many rows have null `amount_eur`. What % of total?
- Why is this a real-world problem? How would you solve it in production?
- What join strategy does Spark pick for `exch_rt_df` (tiny) vs `countries_df` (250 rows)? Check with `.explain()`

In [29]:
countries_df.show(5, truncate=True)

+------------+--------------------+------------+--------------+--------------------+--------------+-------+---------------+--------+----------+
|country_name|       official_name|country_code|currency_codes|      language_codes|       capital| region|      subregion|    area|population|
+------------+--------------------+------------+--------------+--------------------+--------------+-------+---------------+--------+----------+
|    Zimbabwe|Republic of Zimbabwe|          ZW|         [ZWL]|[sna, ndc, khi, t...|      [Harare]| Africa|Southern Africa|390757.0|  17073087|
|    Kiribati|Independent and S...|          KI|    [AUD, KID]|          [gil, eng]|[South Tarawa]|Oceania|     Micronesia|   811.0|    120740|
|       Ghana|   Republic of Ghana|          GH|         [GHS]|               [eng]|       [Accra]| Africa| Western Africa|238533.0|  33742380|
| North Korea|Democratic People...|          KP|         [KPW]|               [kor]|   [Pyongyang]|   Asia|   Eastern Asia|120538.0|  25

In [30]:
exch_rt_df.show(5, truncate=True)

+----------+-------------+---------------+-------+
| rate_date|base_currency|target_currency|   rate|
+----------+-------------+---------------+-------+
|2024-10-01|          EUR|            AUD|  1.604|
|2024-10-01|          EUR|            CAD| 1.4986|
|2024-10-01|          EUR|            CHF| 0.9394|
|2024-10-01|          EUR|            CZK| 25.272|
|2024-10-01|          EUR|            GBP|0.83193|
+----------+-------------+---------------+-------+
only showing top 5 rows



In [31]:
enriched_transactions = (
    transactions.alias("tx").join(
        countries_df.alias("ctry"), 
        F.col("tx.country_code") == F.col("ctry.country_code"), 
        "left")
).select("tx.*", "ctry.country_name", "ctry.subregion", "ctry.population")

enriched_transactions = (
    enriched_transactions.alias("tx").join(
        exch_rt_df.alias("ex"), 
        (F.col("tx.currency")==F.col("ex.target_currency")) 
            & (F.col("tx.txn_date")==F.col("ex.rate_date")), 
        "left")
    .withColumn("amount_eur", F.col("amount") / F.col("rate"))
).select("txn_id", "customer_id", "merchant_name", "category", "amount", "currency", "amount_eur", "country_name", "region", "population", "status", "txn_date", "fraud_score")

enriched_transactions.show(5, truncate=True)
enriched_transactions.explain()

+----------+-----------+-------------+---------+------+--------+----------+------------+--------+----------+---------+----------+-----------+
|    txn_id|customer_id|merchant_name| category|amount|currency|amount_eur|country_name|  region|population|   status|  txn_date|fraud_score|
+----------+-----------+-------------+---------+------+--------+----------+------------+--------+----------+---------+----------+-----------+
|TXN-000002|     C-0130|       Amazon|   retail|422.21|     KES|      NULL|       Kenya|  Africa|  53330978|completed|2025-02-17|      0.529|
|TXN-000001|     C-0058|         Uber|transport|  99.4|     NPR|      NULL|       Nepal|    Asia|  29911840|completed|2025-02-17|     0.5623|
|TXN-000008|     C-0019|    Carrefour|  grocery|427.17|     XOF|      NULL|        Mali|  Africa|  22395489|completed|2025-02-14|       NULL|
|TXN-000003|     C-0002|         Bolt|transport|531.65|     ZMW|      NULL|      Zambia|  Africa|  19693423|completed|2025-01-17|       NULL|
|TXN-0

In [32]:
enriched_transactions.select(
    F.count("*").alias("total"),
    F.sum(F.col("amount_eur").isNull().cast("int")).alias("null_count"),
    (F.sum(F.col("amount_eur").isNull().cast("int")) / F.count("*") * 100).alias("null_pct")
).show()

+-----+----------+-----------------+
|total|null_count|         null_pct|
+-----+----------+-----------------+
|  869|       808|92.98043728423475|
+-----+----------+-----------------+



#### Exercise 3: Window Functions (15 min)

Use your enriched view from Exercise 2, or just `transactions` + `countries_df`.

1. **Ranking:** Per `country_name`, rank transactions by `amount` DESC.
   Column: `country_amount_rank`. Keep only top 5 per country.

2. **Running total:** Per `customer_id`, running total of `amount` ordered by `txn_date`.
   Column: `customer_running_total`

3. **Lag:** Per `customer_id`, days between current txn and previous txn (by date).
   Column: `days_since_prev_txn` (null for first txn per customer)

4. **Percent of total:** Per `region`, each txn's amount as a % of the region's total amount.
   Column: `pct_of_region_total`

5. **Dense rank:** Rank countries by total txn count, WITHIN their region.
   Return: `region`, `country_name`, `txn_count`, `rank_in_region`

**After you code it:**

- Q2: Is your running total deterministic if two txns share a date?
- Q1: ROW_NUMBER vs RANK — which did you use? What happens on ties?
- Q4: Window-only approach vs window+join — which is more efficient?

In [33]:
# 3.1 — Top 5 transactions per country by amount
w_country = Window.partitionBy("country_name").orderBy(F.desc("amount"))

top5_per_country = (
    enriched_transactions
    .withColumn("country_amount_rank", F.row_number().over(w_country))
    .filter(F.col("country_amount_rank") <= 5)
)
top5_per_country.select("country_name", "txn_id", "amount", "currency", "country_amount_rank").show(15, truncate=False)

+------------+----------+------+--------+-------------------+
|country_name|txn_id    |amount|currency|country_amount_rank|
+------------+----------+------+--------+-------------------+
|Afghanistan |TXN-000777|665.05|AFN     |1                  |
|Afghanistan |TXN-000877|500.07|AFN     |2                  |
|Afghanistan |TXN-000505|374.18|AFN     |3                  |
|Afghanistan |TXN-001049|99.9  |AFN     |4                  |
|Albania     |TXN-000687|261.37|ALL     |1                  |
|Algeria     |TXN-000073|675.38|DZD     |1                  |
|Algeria     |TXN-000257|639.13|DZD     |2                  |
|Algeria     |TXN-000020|529.18|DZD     |3                  |
|Algeria     |TXN-001362|460.3 |DZD     |4                  |
|Algeria     |TXN-000614|459.71|DZD     |5                  |
|Angola      |TXN-000874|577.43|AOA     |1                  |
|Angola      |TXN-000780|489.24|AOA     |2                  |
|Angola      |TXN-000174|475.51|AOA     |3                  |
|Angola 

In [34]:
# 3.2 — Running total per customer ordered by txn_date
w_running = Window.partitionBy("customer_id").orderBy("txn_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_totals = enriched_transactions.withColumn(
    "customer_running_total", F.sum("amount").over(w_running)
)
running_totals.select("customer_id", "txn_date", "amount", "customer_running_total").show(15)

+-----------+----------+------+----------------------+
|customer_id|  txn_date|amount|customer_running_total|
+-----------+----------+------+----------------------+
|     C-0001|2025-01-19| 214.1|                 214.1|
|     C-0001|2025-01-24|530.44|     744.5400000000001|
|     C-0001|2025-01-24|691.79|               1436.33|
|     C-0001|2025-01-25|689.75|               2126.08|
|     C-0002|2024-10-11| 293.6|                 293.6|
|     C-0002|2024-11-20|237.31|     530.9100000000001|
|     C-0002|2025-01-14|384.65|     915.5600000000001|
|     C-0002|2025-01-15|369.16|               1284.72|
|     C-0002|2025-01-17|531.65|               1816.37|
|     C-0002|2025-01-23|184.03|    2000.3999999999999|
|     C-0002|2025-01-29|563.45|               2563.85|
|     C-0002|2025-02-19|372.19|               2936.04|
|     C-0002|2025-02-21|189.17|               3125.21|
|     C-0002|2025-02-25|496.99|                3622.2|
|     C-0003|2024-10-04| 164.5|                 164.5|
+---------

In [35]:
# 3.3 — Days since previous transaction per customer
w_lag = Window.partitionBy("customer_id").orderBy("txn_date")

with_lag = enriched_transactions.withColumn(
    "prev_txn_date", F.lag("txn_date").over(w_lag)
).withColumn(
    "days_since_prev_txn", F.datediff("txn_date", "prev_txn_date")
)
with_lag.select("customer_id", "txn_date", "prev_txn_date", "days_since_prev_txn").show(15)

+-----------+----------+-------------+-------------------+
|customer_id|  txn_date|prev_txn_date|days_since_prev_txn|
+-----------+----------+-------------+-------------------+
|     C-0001|2025-01-19|         NULL|               NULL|
|     C-0001|2025-01-24|   2025-01-19|                  5|
|     C-0001|2025-01-24|   2025-01-24|                  0|
|     C-0001|2025-01-25|   2025-01-24|                  1|
|     C-0002|2024-10-11|         NULL|               NULL|
|     C-0002|2024-11-20|   2024-10-11|                 40|
|     C-0002|2025-01-14|   2024-11-20|                 55|
|     C-0002|2025-01-15|   2025-01-14|                  1|
|     C-0002|2025-01-17|   2025-01-15|                  2|
|     C-0002|2025-01-23|   2025-01-17|                  6|
|     C-0002|2025-01-29|   2025-01-23|                  6|
|     C-0002|2025-02-19|   2025-01-29|                 21|
|     C-0002|2025-02-21|   2025-02-19|                  2|
|     C-0002|2025-02-25|   2025-02-21|                  

In [36]:
# 3.4 — Each transaction's amount as % of region total
w_region = Window.partitionBy("region")

with_pct = enriched_transactions.withColumn(
    "region_total", F.sum("amount").over(w_region)
).withColumn(
    "pct_of_region_total", F.round(F.col("amount") / F.col("region_total") * 100, 4)
).drop("region_total")

with_pct.select("region", "txn_id", "amount", "pct_of_region_total").show(10)

+------+----------+------+-------------------+
|region|    txn_id|amount|pct_of_region_total|
+------+----------+------+-------------------+
|Africa|TXN-000002|422.21|             0.3943|
|Africa|TXN-000011| 99.46|             0.0929|
|Africa|TXN-000014|476.84|             0.4453|
|Africa|TXN-000020|529.18|             0.4941|
|Africa|TXN-000073|675.38|             0.6307|
|Africa|TXN-000114| 54.23|             0.0506|
|Africa|TXN-000148| 48.86|             0.0456|
|Africa|TXN-000150|247.45|             0.2311|
|Africa|TXN-000177|356.92|             0.3333|
|Africa|TXN-000183|561.28|             0.5241|
+------+----------+------+-------------------+
only showing top 10 rows



In [37]:
# 3.5 — Rank countries by txn count within their region
country_txn_counts = (
    enriched_transactions
    .groupBy("region", "country_name")
    .agg(F.count("*").alias("txn_count"))
)

w_region_rank = Window.partitionBy("region").orderBy(F.desc("txn_count"))

country_ranks = country_txn_counts.withColumn(
    "rank_in_region", F.dense_rank().over(w_region_rank)
)
country_ranks.orderBy("region", "rank_in_region").show(30, truncate=False)


+------+------------------------+---------+--------------+
|region|country_name            |txn_count|rank_in_region|
+------+------------------------+---------+--------------+
|Africa|Namibia                 |15       |1             |
|Africa|Lesotho                 |11       |2             |
|Africa|Mali                    |11       |2             |
|Africa|Sudan                   |9        |3             |
|Africa|Eritrea                 |9        |3             |
|Africa|Equatorial Guinea       |9        |3             |
|Africa|Eswatini                |9        |3             |
|Africa|Algeria                 |8        |4             |
|Africa|Togo                    |8        |4             |
|Africa|Guinea-Bissau           |8        |4             |
|Africa|Chad                    |8        |4             |
|Africa|Gabon                   |8        |4             |
|Africa|Morocco                 |7        |5             |
|Africa|Botswana                |7        |5            

#### Exercise 4: UDF vs Native Spark (10 min)

Risk classification rules:

| Priority | Label | Condition |
|---|---|---|
| 1 | critical | `fraud_score > 0.85` AND `amount > 400` |
| 2 | high | `fraud_score > 0.85` OR (`amount > 600` AND `category` in `("retail", "delivery")`) |
| 3 | medium | `fraud_score > 0.5` OR `amount > 300` |
| 4 | low | everything else |

`fraud_score` is NULL for ~94% of rows. NULL must NOT trigger fraud-based rules.

1. Write it as a Python `@udf`. Apply it. Show sample results.
2. Rewrite as pure `F.when().otherwise()` chains. Apply it.
3. Verify both produce identical results.

**Explain (in comments or out loud):**

- What happens at JVM level with the Python UDF? (serialization, worker-side Python, Catalyst bypass)
- Why is native faster?
- Does `when()` order matter? What if you check "low" first?


In [38]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# 4.1 — Python UDF
@udf(StringType())
def classify_risk_udf(fraud_score, amount, category):
    # fraud_score is None for most rows
    fs = fraud_score if fraud_score is not None else -1.0
    if fs > 0.85 and amount > 400:
        return "critical"
    elif fs > 0.85 or (amount > 600 and category in ("retail", "delivery")):
        return "high"
    elif fs > 0.5 or amount > 300:
        return "medium"
    else:
        return "low"

with_risk_udf = enriched_transactions.withColumn(
    "risk_udf", classify_risk_udf("fraud_score", "amount", "category")
)
with_risk_udf.select("txn_id", "amount", "category", "fraud_score", "risk_udf").show(10)

+----------+------+------------+-----------+--------+
|    txn_id|amount|    category|fraud_score|risk_udf|
+----------+------+------------+-----------+--------+
|TXN-000002|422.21|      retail|      0.529|  medium|
|TXN-000011| 99.46|     grocery|       NULL|     low|
|TXN-000014|476.84|        fuel|       NULL|  medium|
|TXN-000015|112.39|    delivery|       NULL|     low|
|TXN-000016|589.49|      retail|       NULL|  medium|
|TXN-000012|357.65|subscription|       NULL|  medium|
|TXN-000001|  99.4|   transport|     0.5623|  medium|
|TXN-000008|427.17|     grocery|       NULL|  medium|
|TXN-000003|531.65|   transport|       NULL|  medium|
|TXN-000005|511.35|   transport|       NULL|  medium|
+----------+------+------------+-----------+--------+
only showing top 10 rows



In [39]:
# 4.2 — Native Spark (no UDF)
with_risk_native = enriched_transactions.withColumn(
    "risk_native",
    F.when(
        (F.col("fraud_score") > 0.85) & (F.col("amount") > 400), "critical"
    ).when(
        (F.col("fraud_score") > 0.85) |
        ((F.col("amount") > 600) & (F.col("category").isin("retail", "delivery"))),
        "high"
    ).when(
        (F.col("fraud_score") > 0.5) | (F.col("amount") > 300), "medium"
    ).otherwise("low")
)
with_risk_native.select("txn_id", "amount", "category", "fraud_score", "risk_native").show(10)

+----------+------+------------+-----------+-----------+
|    txn_id|amount|    category|fraud_score|risk_native|
+----------+------+------------+-----------+-----------+
|TXN-000002|422.21|      retail|      0.529|     medium|
|TXN-000011| 99.46|     grocery|       NULL|        low|
|TXN-000014|476.84|        fuel|       NULL|     medium|
|TXN-000015|112.39|    delivery|       NULL|        low|
|TXN-000016|589.49|      retail|       NULL|     medium|
|TXN-000012|357.65|subscription|       NULL|     medium|
|TXN-000001|  99.4|   transport|     0.5623|     medium|
|TXN-000008|427.17|     grocery|       NULL|     medium|
|TXN-000003|531.65|   transport|       NULL|     medium|
|TXN-000005|511.35|   transport|       NULL|     medium|
+----------+------+------------+-----------+-----------+
only showing top 10 rows



In [40]:
# 4.3 — Verify identical results
comparison = (
    with_risk_udf.select("txn_id", F.col("risk_udf").alias("udf_result"))
    .join(
        with_risk_native.select("txn_id", F.col("risk_native").alias("native_result")),
        "txn_id"
    )
    .filter(F.col("udf_result") != F.col("native_result"))
)
mismatch_count = comparison.count()
print(f"Mismatches: {mismatch_count}")

Mismatches: 0


#### Exercise 5: Aggregations (10 min)

##### Report 1 — Customer Summary

From `transactions`, completed only. Per `customer_id`:

- `total_txns` — count
- `total_amount` — sum
- `avg_amount` — average
- `distinct_countries` — unique country codes
- `distinct_currencies` — unique currency codes
- `favorite_merchant` — merchant with highest total spend *(the hard one)*
- `days_active` — days between first and last txn
- `fraud_flag_count` — txns where `fraud_score IS NOT NULL`

Order by `total_amount` DESC. Show top 20.

##### Report 2 — Region Currency Coverage

Join `countries_df` with `live_rates` on `currency_code = target`. Per `region`:

- `num_countries` — count of countries
- `total_population` — sum
- `num_currencies` — distinct currency codes in that region
- `num_with_rate` — how many of those currencies appear in `live_rates`
- `coverage_pct` — `num_with_rate / num_currencies * 100`

Order by `total_population` DESC.

**After you code it:**

- How did you solve `favorite_merchant`? Window inside agg? Subquery? Self-join?
- Which regions have worst rate coverage? Why?

In [41]:
# --- Report 1: Customer Summary ---

# Step 1: favorite_merchant per customer (window approach)
w_merch = Window.partitionBy("customer_id").orderBy(F.desc("merchant_spend"))

merchant_spend = (
    enriched_transactions
    .groupBy("customer_id", "merchant_name")
    .agg(F.sum("amount").alias("merchant_spend"))
)
fav_merchant = (
    merchant_spend
    .withColumn("rn", F.row_number().over(w_merch))
    .filter(F.col("rn") == 1)
    .select("customer_id", F.col("merchant_name").alias("favorite_merchant"))
)

# Step 2: all other metrics
customer_summary = (
    enriched_transactions
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("total_txns"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("avg_amount"),
        F.countDistinct("country_name").alias("distinct_countries"),
        F.countDistinct("currency").alias("distinct_currencies"),
        F.datediff(F.max("txn_date"), F.min("txn_date")).alias("days_active"),
        F.sum(F.when(F.col("fraud_score").isNotNull(), 1).otherwise(0)).alias("fraud_flag_count"),
    )
)

# Step 3: join favorite merchant
customer_report = (
    customer_summary
    .join(fav_merchant, "customer_id", "left")
    .orderBy(F.desc("total_amount"))
)
customer_report.show(20, truncate=False)

+-----------+----------+------------+----------+------------------+-------------------+-----------+----------------+-----------------+
|customer_id|total_txns|total_amount|avg_amount|distinct_countries|distinct_currencies|days_active|fraud_flag_count|favorite_merchant|
+-----------+----------+------------+----------+------------------+-------------------+-----------+----------------+-----------------+
|C-0046     |11        |5109.32     |464.48    |10                |10                 |135        |1               |Shell            |
|C-0133     |12        |4911.95     |409.33    |11                |11                 |148        |0               |Zara             |
|C-0011     |10        |4714.42     |471.44    |10                |9                  |136        |1               |Netflix          |
|C-0122     |12        |4376.39     |364.7     |12                |10                 |115        |1               |Glovo            |
|C-0057     |13        |4234.36     |325.72    |13     

In [42]:
# --- Report 1: Customer Summary ---

# Step 1: favorite_merchant per customer (window approach)
w_merch = Window.partitionBy("customer_id").orderBy(F.desc("merchant_spend"))

merchant_spend = (
    enriched_transactions
    .groupBy("customer_id", "merchant_name")
    .agg(F.sum("amount").alias("merchant_spend"))
)
fav_merchant = (
    merchant_spend
    .withColumn("rn", F.row_number().over(w_merch))
    .filter(F.col("rn") == 1)
    .select("customer_id", F.col("merchant_name").alias("favorite_merchant"))
)

# Step 2: all other metrics
customer_summary = (
    enriched_transactions
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("total_txns"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("avg_amount"),
        F.countDistinct("country_name").alias("distinct_countries"),
        F.countDistinct("currency").alias("distinct_currencies"),
        F.datediff(F.max("txn_date"), F.min("txn_date")).alias("days_active"),
        F.sum(F.when(F.col("fraud_score").isNotNull(), 1).otherwise(0)).alias("fraud_flag_count"),
    )
)

# Step 3: join favorite merchant
customer_report = (
    customer_summary
    .join(fav_merchant, "customer_id", "left")
    .orderBy(F.desc("total_amount"))
)
customer_report.show(20, truncate=False)

+-----------+----------+------------+----------+------------------+-------------------+-----------+----------------+-----------------+
|customer_id|total_txns|total_amount|avg_amount|distinct_countries|distinct_currencies|days_active|fraud_flag_count|favorite_merchant|
+-----------+----------+------------+----------+------------------+-------------------+-----------+----------------+-----------------+
|C-0046     |11        |5109.32     |464.48    |10                |10                 |135        |1               |Shell            |
|C-0133     |12        |4911.95     |409.33    |11                |11                 |148        |0               |Zara             |
|C-0011     |10        |4714.42     |471.44    |10                |9                  |136        |1               |Netflix          |
|C-0122     |12        |4376.39     |364.7     |12                |10                 |115        |1               |Glovo            |
|C-0057     |13        |4234.36     |325.72    |13     

#### Exercise 6: Relational Joins — JSONPlaceholder (8 min, bonus)

Classic normalized model: `users` → `posts` → `comments`.

1. Full denormalized view: user + their posts + comments on those posts.

2. Per user: `total_posts`, `total_comments_received`, `avg_comments_per_post`, `most_commented_post_title`.

3. **Self-commenters:** users who commented on their own posts.
   Match `commenter_email = user's email`.
   Show: user name, post title, comment body.

4. **Engagement score:** `(total_comments_received * 2) + total_posts`.
   Rank users by this score.

In [43]:
# 6.1 — Full denormalized view
denormalized = (
    users.alias("u")
    .join(posts.alias("p"), F.col("u.user_id") == F.col("p.user_id"), "inner")
    .join(comments.alias("c"), F.col("p.post_id") == F.col("c.post_id"), "inner")
    .select(
        F.col("u.user_id"), F.col("u.name").alias("user_name"), F.col("u.email").alias("user_email"),
        F.col("p.post_id"), F.col("p.title"),
        F.col("c.comment_id"), F.col("c.commenter_name"), F.col("c.commenter_email"), F.col("c.comment_body")
    )
)
denormalized.show(5, truncate=True)


+-------+------------+-----------------+-------+--------------------+----------+--------------------+--------------------+--------------------+
|user_id|   user_name|       user_email|post_id|               title|comment_id|      commenter_name|     commenter_email|        comment_body|
+-------+------------+-----------------+-------+--------------------+----------+--------------------+--------------------+--------------------+
|      2|Ervin Howell|Shanna@melissa.tv|     11|et ea vero quia l...|        55|quas pariatur qui...|Hayden_Olson@mari...|perferendis eaque...|
|      2|Ervin Howell|Shanna@melissa.tv|     11|et ea vero quia l...|        54|culpa eius tempor...|Kenton_Vandervort...|et ipsa rem ullam...|
|      2|Ervin Howell|Shanna@melissa.tv|     11|et ea vero quia l...|        53|maiores alias nec...|Laverne_Price@sco...|a at tempore mole...|
|      2|Ervin Howell|Shanna@melissa.tv|     11|et ea vero quia l...|        52|  esse autem dolorum|Abigail.OConnell@...|et enim volupt

In [44]:
# 6.2 — Per-user stats
comments_per_post = (
    posts.alias("p")
    .join(comments.alias("c"), F.col("p.post_id") == F.col("c.post_id"), "left")
    .groupBy("p.post_id", "p.user_id", "p.title")
    .agg(F.count("c.comment_id").alias("comment_count"))
)

w_most_commented = Window.partitionBy("user_id").orderBy(F.desc("comment_count"))

most_commented_post = (
    comments_per_post
    .withColumn("rn", F.row_number().over(w_most_commented))
    .filter(F.col("rn") == 1)
    .select("user_id", F.col("title").alias("most_commented_post_title"))
)

user_stats = (
    comments_per_post
    .groupBy("user_id")
    .agg(
        F.count("post_id").alias("total_posts"),
        F.sum("comment_count").alias("total_comments_received"),
        F.round(F.avg("comment_count"), 2).alias("avg_comments_per_post"),
    )
    .join(most_commented_post, "user_id", "left")
    .join(users.select("user_id", "name"), "user_id", "left")
    .select("user_id", "name", "total_posts", "total_comments_received", "avg_comments_per_post", "most_commented_post_title")
)
user_stats.show(10, truncate=False)


+-------+------------------------+-----------+-----------------------+---------------------+--------------------------------------------------------------------+
|user_id|name                    |total_posts|total_comments_received|avg_comments_per_post|most_commented_post_title                                           |
+-------+------------------------+-----------+-----------------------+---------------------+--------------------------------------------------------------------+
|2      |Ervin Howell            |10         |50                     |5.0                  |in quibusdam tempore odit est dolorem                               |
|4      |Patricia Lebsack        |10         |50                     |5.0                  |ullam ut quidem id aut vel consequuntur                             |
|5      |Chelsey Dietrich        |10         |50                     |5.0                  |commodi ullam sint et excepturi error explicabo praesentium voluptas|
|8      |Nicholas Runolfsdot

In [45]:
# 6.3 — Self-commenters
self_commenters = (
    denormalized
    .filter(F.col("user_email") == F.col("commenter_email"))
    .select("user_name", "title", "comment_body")
)
print(f"Self-comments found: {self_commenters.count()}")
self_commenters.show(truncate=False)

Self-comments found: 0
+---------+-----+------------+
|user_name|title|comment_body|
+---------+-----+------------+
+---------+-----+------------+



In [46]:
# 6.4 — Engagement score
engagement = (
    user_stats
    .withColumn("engagement_score", F.col("total_comments_received") * 2 + F.col("total_posts"))
    .withColumn("engagement_rank", F.row_number().over(Window.orderBy(F.desc("engagement_score"))))
    .select("engagement_rank", "name", "total_posts", "total_comments_received", "engagement_score")
    .orderBy("engagement_rank")
)
engagement.show(10, truncate=False)

26/03/10 15:44:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 15:44:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 15:44:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 15:44:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 15:44:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 15:44:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/10 1

+---------------+------------------------+-----------+-----------------------+----------------+
|engagement_rank|name                    |total_posts|total_comments_received|engagement_score|
+---------------+------------------------+-----------+-----------------------+----------------+
|1              |Ervin Howell            |10         |50                     |110             |
|2              |Patricia Lebsack        |10         |50                     |110             |
|3              |Chelsey Dietrich        |10         |50                     |110             |
|4              |Nicholas Runolfsdottir V|10         |50                     |110             |
|5              |Leanne Graham           |10         |50                     |110             |
|6              |Mrs. Dennis Schulist    |10         |50                     |110             |
|7              |Glenna Reichert         |10         |50                     |110             |
|8              |Clementina DuBuque     